# Public Data Regeneration

# Result Database

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import numpy as np
import importlib
import pandas as pd
import seaborn
from IPython.display import Image
import matplotlib.pyplot as plt
import time

import Transformer as tnsf
import detection_model as ad

importlib.reload(ad)
importlib.reload(tnsf)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [2]:
pd.options.display.width = 400
pd.options.display.max_colwidth = 400

In [ ]:
def build_prediction_database(
    model,
    hdfs_train,
    test_normal,
    test_abnormal,
    top_k=9,
    out_csv="transformer_prediction_database.csv",
    print_every=1000,
    verbose=True
):
    model.eval()
    rows = []
    start_time = time.time()

    # Sequence-level counters
    TP_seq, TN_seq, FP_seq, FN_seq = 0, 0, 0, 0

    # Make sure column name is sequence
    for df in [hdfs_train, test_normal, test_abnormal]:
        if "sequence" not in df.columns:
            df.columns = ["sequence"]

    # Training EventIDs for unseen event detection
    train_events = set(
        hdfs_train["sequence"]
        .astype(str)
        .str.split()
        .explode()
        .astype(int)
        .unique()
    )

    datasets = {
        "train": hdfs_train,
        "normal_test": test_normal,
        "abnormal_test": test_abnormal,
    }

    def compute_metrics(tp, tn, fp, fn):
        total = tp + tn + fp + fn
        if total == 0:
            return 0, 0, 0, 0

        accuracy = 100 * (tp + tn) / total
        precision = 100 * tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = 100 * tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )
        return accuracy, precision, recall, f1

    if verbose:
        print("Train sequences:", len(hdfs_train))
        print("Normal test sequences:", len(test_normal))
        print("Abnormal test sequences:", len(test_abnormal))
        print("Train EventIDs:", [int(x) for x in sorted(train_events)])
        print("-" * 60)

    with torch.no_grad():
        for dataset_name, df in datasets.items():

            if verbose:
                print(f"Processing {dataset_name}: {len(df)} sequences")

            for seq_count, (seq_id, sequence) in enumerate(df["sequence"].astype(str).items()):

                if verbose and seq_count % print_every == 0:
                    print(f"{dataset_name}: {seq_count}")

                seq = list(map(int, sequence.split()))
                input_seq = seq[:WINDOW_SIZE]
                targets = seq[WINDOW_SIZE:]

                if len(input_seq) < WINDOW_SIZE or len(targets) == 0:
                    continue

                src = torch.zeros((1, WINDOW_SIZE + 1), dtype=torch.long).to(device)
                src[0, 0] = BOS
                src[0, 1:] = torch.tensor(input_seq, dtype=torch.long).to(device)

                src_mask = (src != 0).unsqueeze(-2)
                memory = model.encode(src, src_mask)

                ys = torch.ones(1, 1).fill_(BOS).type_as(src.data).to(device)

                sequence_has_anomaly = False
                sequence_rows = []

                for step, target_event in enumerate(targets[:WINDOW_SIZE], start=1):

                    out = model.decode(
                        memory,
                        src_mask,
                        Variable(ys),
                        Variable(subsequent_mask(ys.size(1)).type_as(src.data)).to(device)
                    )

                    log_probs = model.generator(out[:, -1])
                    probs = torch.exp(log_probs).detach().cpu().numpy()[0]

                    # Exclude PAD and BOS from prediction candidates
                    probs[0] = -1
                    probs[BOS] = -1

                    pred_events = np.argsort(probs)[-top_k:][::-1].tolist()
                    pred_probs = [float(probs[e]) for e in pred_events]

                    target_in_topk = target_event in pred_events
                    target_pos = pred_events.index(target_event) + 1 if target_in_topk else None
                    target_prob = float(probs[target_event]) if target_in_topk else None

                    predicted_status = "normal" if target_in_topk else "abnormal"

                    if not target_in_topk:
                        sequence_has_anomaly = True

                    # Token-level confusion matrix
                    if dataset_name == "train":
                        row_results_matrix = None

                    elif dataset_name == "normal_test":
                        row_results_matrix = (
                            "True Negative" if target_in_topk else "False Positive"
                        )

                    elif dataset_name == "abnormal_test":
                        row_results_matrix = (
                            "False Negative" if target_in_topk else "True Positive"
                        )

                    row = {
                        "dataset": dataset_name,
                        "seq_id": seq_id,
                        "sequence": sequence,
                        "seq_len": len(seq),
                        "input_seq": input_seq,
                        "target_seq": targets[:step],
                        "step": step,
                        "pred_events": pred_events,
                        "pred_probs": pred_probs,
                        "target_event": target_event,
                        "target_event_is_unseen": target_event not in train_events,
                        "target_in_topk_pred": target_in_topk,
                        "target_in_pred_pos": target_pos,
                        "target_in_topk_pred_prob": target_prob,
                        "predicted_status": predicted_status,
                        "results": target_in_topk,
                        "results_matrix": row_results_matrix,
                    }

                    sequence_rows.append(row)

                    # Teacher forcing: Append actual target event
                    ys = torch.cat(
                        [
                            ys,
                            torch.ones(1, 1)
                            .type_as(src.data)
                            .fill_(target_event)
                            .to(device),
                        ],
                        dim=1,
                    )

                # Sequence-level confusion matrix
                if dataset_name == "train":
                    sequence_results_matrix = None

                elif dataset_name == "normal_test":
                    if sequence_has_anomaly:
                        FP_seq += 1
                        sequence_results_matrix = "False Positive"
                    else:
                        TN_seq += 1
                        sequence_results_matrix = "True Negative"

                elif dataset_name == "abnormal_test":
                    if sequence_has_anomaly:
                        TP_seq += 1
                        sequence_results_matrix = "True Positive"
                    else:
                        FN_seq += 1
                        sequence_results_matrix = "False Negative"

                for row in sequence_rows:
                    row["sequence_predicted_status"] = (
                        "abnormal" if sequence_has_anomaly else "normal"
                    )
                    row["sequence_results_matrix"] = sequence_results_matrix
                    rows.append(row)

            if verbose:
                print(f"Finished {dataset_name}")
                print("-" * 60)

    # Dataframe
    result_df = pd.DataFrame(rows)

    # Save CSV
    result_df.to_csv(out_csv, index=False)

    # -----------------------------
    # Token-level evaluation
    # -----------------------------
    eval_rows = result_df[result_df["dataset"].isin(["normal_test", "abnormal_test"])]

    row_counts = eval_rows["results_matrix"].value_counts()

    TP_row = int(row_counts.get("True Positive", 0))
    TN_row = int(row_counts.get("True Negative", 0))
    FP_row = int(row_counts.get("False Positive", 0))
    FN_row = int(row_counts.get("False Negative", 0))

    A_row, P_row, R_row, F1_row = compute_metrics(TP_row, TN_row, FP_row, FN_row)

    # -----------------------------
    # Sequence-level evaluation
    # -----------------------------
    A_seq, P_seq, R_seq, F1_seq = compute_metrics(TP_seq, TN_seq, FP_seq, FN_seq)

    elapsed_time = time.time() - start_time

    if verbose:
        print("CSV saved to:", out_csv)
        print("Total exported prediction rows:", len(result_df))
        print("Evaluation prediction rows:", len(eval_rows))
        print()

        print("Prediction-row-level evaluation")
        print("TP:", TP_row)
        print("TN:", TN_row)
        print("FP:", FP_row)
        print("FN:", FN_row)
        print(f"Accuracy: {A_row:.3f}%")
        print(f"Precision: {P_row:.3f}%")
        print(f"Recall: {R_row:.3f}%")
        print(f"F1-measure: {F1_row:.3f}%")
        print()

        print("Sequence-level evaluation")
        print("TP:", TP_seq)
        print("TN:", TN_seq)
        print("FP:", FP_seq)
        print("FN:", FN_seq)
        print(f"Accuracy: {A_seq:.3f}%")
        print(f"Precision: {P_seq:.3f}%")
        print(f"Recall: {R_seq:.3f}%")
        print(f"F1-measure: {F1_seq:.3f}%")
        print()

        print("elapsed_time: {:.3f}s".format(elapsed_time))

    return result_df

In [10]:
model = tnsf.make_model(
    tnsf.VOCAB_SIZE,
    tnsf.VOCAB_SIZE,
    N=4,
    d_model=512,
    d_ff=2048,
    h=4,
    dropout=0.1
)

model.load_state_dict(
    torch.load(
        r"C:\01. Monash University\controlroom-xai-alarm-support-transformer\notebooks\Model\centralized_model.pt",
        map_location=device
    )
)

model = model.to(device)
model.eval()

EncoderDecoder(
  (encoder): Encoder(
    (layers): ModuleList(
      (0-3): 4 x EncoderLayer(
        (self_attn): MultiHeadedAttention(
          (linears): ModuleList(
            (0-3): 4 x Linear(in_features=512, out_features=512, bias=True)
          )
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=512, out_features=2048, bias=True)
          (w_2): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sublayer): ModuleList(
          (0-1): 2 x SublayerConnection(
            (norm): LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
    )
    (norm): LayerNorm()
  )
  (decoder): Decoder(
    (layers): ModuleList(
      (0-3): 4 x DecoderLayer(
        (self_attn): MultiHeadedAttention(
          (linears): ModuleList(
            (0-3): 4 x Linear(in_features=512, out_

In [11]:
WINDOW_SIZE = tnsf.WINDOW_SIZE
BOS = tnsf.BOS
subsequent_mask = tnsf.subsequent_mask

In [5]:
hdfs_train = pd.read_csv(
    r"C:\01. Monash University\controlroom-xai-alarm-support-transformer\notebooks\Dataset\HDFS\hdfs_train",
    header=None,
    names=["sequence"]
)

hdfs_normal = pd.read_csv(
    r"C:\01. Monash University\controlroom-xai-alarm-support-transformer\notebooks\Dataset\HDFS\hdfs_test_normal",
    header=None,
    names=["sequence"]
)

hdfs_abnormal = pd.read_csv(
    r"C:\01. Monash University\controlroom-xai-alarm-support-transformer\notebooks\Dataset\HDFS\hdfs_test_abnormal",
    header=None,
    names=["sequence"]
)

In [36]:
prediction_db = build_prediction_database(
    model,
    hdfs_train,
    hdfs_normal,
    hdfs_abnormal,
    top_k=9,
    out_csv="prediction_db.csv"
)

Train sequences: 4982
Normal test sequences: 553241
Abnormal test sequences: 16838
Train EventIDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 23, 25, 32]
------------------------------------------------------------
Processing train: 4982 sequences
train: 0
train: 1000
train: 2000
train: 3000
train: 4000
Finished train
------------------------------------------------------------
Processing normal_test: 553241 sequences
normal_test: 0
normal_test: 1000
normal_test: 2000
normal_test: 3000
normal_test: 4000
normal_test: 5000
normal_test: 6000
normal_test: 7000
normal_test: 8000
normal_test: 9000
normal_test: 10000
normal_test: 11000
normal_test: 12000
normal_test: 13000
normal_test: 14000
normal_test: 15000
normal_test: 16000
normal_test: 17000
normal_test: 18000
normal_test: 19000
normal_test: 20000
normal_test: 21000
normal_test: 22000
normal_test: 23000
normal_test: 24000
normal_test: 25000
normal_test: 26000
normal_test: 27000
normal_test: 28000
normal_test: 29000
normal_test: 30000
normal_test: 

In [39]:
prediction_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 4679295 entries, 0 to 4679294
Data columns (total 18 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   dataset                    str    
 1   seq_id                     int64  
 2   sequence                   str    
 3   seq_len                    int64  
 4   input_seq                  object 
 5   target_seq                 object 
 6   pred_events                object 
 7   pred_probs                 object 
 8   target_event               int64  
 9   target_event_is_unseen     bool   
 10  target_in_topk_pred        bool   
 11  target_in_pred_pos         float64
 12  target_in_topk_pred_prob   float64
 13  predicted_status           str    
 14  results                    bool   
 15  results_matrix             str    
 16  sequence_predicted_status  str    
 17  sequence_results_matrix    str    
dtypes: bool(3), float64(2), int64(3), object(4), str(6)
memory usage: 976.3+ MB


In [40]:
prediction_db[prediction_db['target_event_is_unseen'] == True].head()

,dataset,seq_id,sequence,seq_len,input_seq,target_seq,pred_events,pred_probs,target_event,target_event_is_unseen,target_in_topk_pred,target_in_pred_pos,target_in_topk_pred_prob,predicted_status,results,results_matrix,sequence_predicted_status,sequence_results_matrix
42803,normal_test,216,2 1 1 1 5 5 3 4 3 4 3 4 17 5 25 25 25 25 25 6 25 25 25 25 25 25 25 25 25 25 23 23 23 13 13 13,36,"[2, 1, 1, 1, 5, 5, 3, 4, 3, 4]","[3, 4, 17]","[3, 23, 25, 5, 4, 9, 15, 26, 24]","[0.42939114570617676, 0.26261264085769653, 0.2507931590080261, 0.053703539073467255, 0.002202547388151288, 0.0005940991104580462, 8.037125371629372e-05, 7.872283458709717e-05, 6.734355702064931e-05]",17,True,False,NaN,NaN,abnormal,False,False Positive,abnormal,False Positive
42824,normal_test,218,2 1 1 1 5 5 3 4 3 4 3 4 5 17 25 25 25 6 25 25 25 25 25 25 25 25 25 25 25 25 23 23 23 13 13 13,36,"[2, 1, 1, 1, 5, 5, 3, 4, 3, 4]","[3, 4, 5, 17]","[23, 9, 25, 13, 4, 6, 5, 24, 8]","[0.975217342376709, 0.022788768634200096, 0.0018996645230799913, 4.4215299567440525e-05, 3.6974284739699215e-05, 3.07184404846339e-06, 2.289129270138801e-06, 1.0752585239970358e-06, 7.434401254613476e-07]",17,True,False,NaN,NaN,abnormal,False,False Positive,abnormal,False Positive
4575479,abnormal_test,213,2 1 1 1 5 5 5 14 3 4 14 3 4 14 3 4 9 6 9 9 23 23 23 13 13 13,26,"[2, 1, 1, 1, 5, 5, 5, 14, 3, 4]",[14],"[4, 3, 9, 5, 25, 24, 22, 12, 2]","[0.9911023378372192, 0.008896291255950928, 6.090891133680998e-07, 2.3432099283127172e-07, 2.0595739158579818e-07, 1.745996769386693e-07, 2.7747244502052126e-08, 2.7482730757810714e-08, 2.0352034724169243e-08]",14,True,False,NaN,NaN,abnormal,False,True Positive,abnormal,True Positive
4575482,abnormal_test,213,2 1 1 1 5 5 5 14 3 4 14 3 4 14 3 4 9 6 9 9 23 23 23 13 13 13,26,"[2, 1, 1, 1, 5, 5, 5, 14, 3, 4]","[14, 3, 4, 14]","[23, 25, 3, 5, 9, 4, 13, 15, 24]","[0.6048964858055115, 0.33269578218460083, 0.037958379834890366, 0.022251581773161888, 0.000974657479673624, 0.0007775476551614702, 6.405940075637773e-05, 4.829705721931532e-05, 4.3689218728104606e-05]",14,True,False,NaN,NaN,abnormal,False,True Positive,abnormal,True Positive
4575519,abnormal_test,217,2 1 1 1 5 5 5 14 3 4 14 3 4 14 3 4 6 9 9 23 23 23 13 13 13,25,"[2, 1, 1, 1, 5, 5, 5, 14, 3, 4]",[14],"[4, 3, 9, 5, 25, 24, 22, 12, 2]","[0.9911023378372192, 0.008896291255950928, 6.090891133680998e-07, 2.3432099283127172e-07, 2.0595739158579818e-07, 1.745996769386693e-07, 2.7747244502052126e-08, 2.7482730757810714e-08, 2.0352034724169243e-08]",14,True,False,NaN,NaN,abnormal,False,True Positive,abnormal,True Positive


In [42]:
import Transformer_original as tnsf_original

In [44]:
# Model
model_ori = tnsf_original.make_model(
    tnsf_original.VOCAB_SIZE,
    tnsf_original.VOCAB_SIZE,
    N=4,
    d_model=512,
    d_ff=2048,
    h=4,
    dropout=0.1
)

model_ori.load_state_dict(
    torch.load(
        r"C:\01. Monash University\controlroom-xai-alarm-support-transformer\notebooks\Model\original_model.pt",
        map_location=device
    )
)

model_ori = model_ori.to(device)
model_ori.eval()

EncoderDecoder(
  (encoder): Encoder(
    (layers): ModuleList(
      (0-3): 4 x EncoderLayer(
        (self_attn): MultiHeadedAttention(
          (linears): ModuleList(
            (0-3): 4 x Linear(in_features=512, out_features=512, bias=True)
          )
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=512, out_features=2048, bias=True)
          (w_2): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sublayer): ModuleList(
          (0-1): 2 x SublayerConnection(
            (norm): LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
    )
    (norm): LayerNorm()
  )
  (decoder): Decoder(
    (layers): ModuleList(
      (0-3): 4 x DecoderLayer(
        (self_attn): MultiHeadedAttention(
          (linears): ModuleList(
            (0-3): 4 x Linear(in_features=512, out_

In [45]:
# Parameters
WINDOW_SIZE = tnsf_original.WINDOW_SIZE
BOS = tnsf_original.BOS
subsequent_mask = tnsf_original.subsequent_mask

In [46]:
prediction_db_ori = build_prediction_database(
    model_ori,
    hdfs_train,
    hdfs_normal,
    hdfs_abnormal,
    top_k=9,
    out_csv="prediction_db_ori.csv"
)

Train sequences: 4982
Normal test sequences: 553241
Abnormal test sequences: 16838
Train EventIDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 23, 25, 32]
------------------------------------------------------------
Processing train: 4982 sequences
train: 0
train: 1000
train: 2000
train: 3000
train: 4000
Finished train
------------------------------------------------------------
Processing normal_test: 553241 sequences
normal_test: 0
normal_test: 1000
normal_test: 2000
normal_test: 3000
normal_test: 4000
normal_test: 5000
normal_test: 6000
normal_test: 7000
normal_test: 8000
normal_test: 9000
normal_test: 10000
normal_test: 11000
normal_test: 12000
normal_test: 13000
normal_test: 14000
normal_test: 15000
normal_test: 16000
normal_test: 17000
normal_test: 18000
normal_test: 19000
normal_test: 20000
normal_test: 21000
normal_test: 22000
normal_test: 23000
normal_test: 24000
normal_test: 25000
normal_test: 26000
normal_test: 27000
normal_test: 28000
normal_test: 29000
normal_test: 30000
normal_test: 

# Transformer Performance Token-Level


In [4]:
prediction_db = pd.read_csv("prediction_db.csv")
prediction_db_ori = pd.read_csv("prediction_db_ori.csv")

C:\Users\miqba\AppData\Local\Temp\ipykernel_17284\1819591545.py:1: DtypeWarning: Columns (0: results_matrix, 1: sequence_results_matrix) have mixed types. Specify dtype option on import or set low_memory=False.
  prediction_db = pd.read_csv("prediction_db.csv")
C:\Users\miqba\AppData\Local\Temp\ipykernel_17284\1819591545.py:2: DtypeWarning: Columns (0: results_matrix, 1: sequence_results_matrix) have mixed types. Specify dtype option on import or set low_memory=False.
  prediction_db_ori = pd.read_csv("prediction_db_ori.csv")


In [7]:
prediction_db['dataset'].value_counts()

dataset
normal_test      4533578
abnormal_test     105076
train              40641
Name: count, dtype: int64

In [9]:
prediction_db[prediction_db["dataset"] == "normal_test"].head(5)

,dataset,seq_id,sequence,seq_len,input_seq,target_seq,pred_events,pred_probs,target_event,target_event_is_unseen,target_in_topk_pred,target_in_pred_pos,target_in_topk_pred_prob,predicted_status,results,results_matrix,sequence_predicted_status,sequence_results_matrix
40641,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]",[4],"[4, 3, 9, 5, 25, 24, 22, 12, 2]","[0.9903373718261719, 0.009661218151450157, 6.329992174869403e-07, 2.7030228011426516e-07, 2.1788400772493333e-07, 1.7994237566654192e-07, 2.916568142552478e-08, 2.8556975451010658e-08, 2.150493294550415e-08]",4,False,True,1.0,0.990337,normal,True,True Negative,normal,True Negative
40642,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3]","[3, 4, 5, 25, 23, 26, 24, 2, 28]","[0.9997363686561584, 0.00018246128456667066, 7.214649667730555e-05, 6.90634533384582e-06, 3.9705611243334715e-07, 3.125783223367762e-07, 1.6053230922352668e-07, 1.4491354249912547e-07, 1.0573919695389122e-07]",3,False,True,1.0,0.999736,normal,True,True Negative,normal,True Negative
40643,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3, 4]","[4, 9, 3, 24, 10, 5, 25, 6, 2]","[0.9999996423721313, 2.5044428753062675e-07, 1.1106849484576742e-07, 1.3098173212711117e-08, 1.059248511126043e-08, 1.0226953506276004e-08, 6.82894674142176e-09, 5.357434051944665e-09, 4.159580679896635e-09]",4,False,True,1.0,1.000000,normal,True,True Negative,normal,True Negative
40644,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3, 4, 9]","[23, 25, 9, 5, 3, 6, 4, 13, 15]","[0.5589843988418579, 0.4388941526412964, 0.0009988080710172653, 0.0007600008975714445, 0.00022740353597328067, 3.102449409198016e-05, 2.6064000849146396e-05, 1.656256426940672e-05, 8.248433914559428e-06]",9,False,True,3.0,0.000999,normal,True,True Negative,normal,True Negative
40645,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3, 4, 9, 6]","[23, 9, 25, 4, 13, 6, 24, 12, 8]","[0.8473748564720154, 0.1509784460067749, 0.001563208643347025, 6.786578160244972e-05, 4.2939518607454374e-06, 3.296773684269283e-06, 1.3069163742329692e-06, 9.290149023399863e-07, 8.16922067770065e-07]",6,False,True,6.0,0.000003,normal,True,True Negative,normal,True Negative


In [5]:
prediction_db.groupby("dataset")["results"].value_counts()

dataset        results
abnormal_test  True         88894
               False        16182
normal_test    True       4523680
               False         9898
train          True         40555
               False           86
Name: count, dtype: int64

In [10]:
prediction_db.groupby("dataset")["sequence_results_matrix"].value_counts()

dataset        sequence_results_matrix
abnormal_test  True Positive                84781
               False Negative               20295
normal_test    True Negative              4471870
               False Positive               61708
Name: count, dtype: int64

In [11]:
prediction_db_ori.groupby("dataset")["sequence_results_matrix"].value_counts()

dataset        sequence_results_matrix
abnormal_test  True Positive                75691
               False Negative               29385
normal_test    True Negative              4512130
               False Positive               21448
Name: count, dtype: int64

* Accuracy = (TP + TN) / (TP + FP + FN + TN)
* Precision = TP / (TP + FP)  -> Of all the logs that were predicted as anomalies, how many were actually anomalies?
* Recall = TP / (TP + FN)  -> Of all the actual anomalies, how many were correctly identified?
* F1 Score = 2 * (Precision * Recall) / (Precision + Recall)

In [16]:
# BOS Adjusted Transformer
TP, TN, FP, FN = 84781, 4471870, 61708, 20295

acc = (TP + TN) / (TP + TN + FP + FN) * 100
prec = TP / (TP + FP) * 100
rec = TP / (TP + FN) * 100
f1 = 2 * (prec/100) * (rec/100) / ((prec/100) + (rec/100)) * 100

print("BOS Adjusted Transformer:")
print(f"Accuracy: {acc:.2f}")
print(f"Precision: {prec:.2f}")
print(f"Recall: {rec:.2f}")
print(f"F1-measure: {f1:.2f}")

# Original Transformer
TP, TN, FP, FN = 75691, 4512130, 21448, 29385

acc = (TP + TN) / (TP + TN + FP + FN) * 100
prec = TP / (TP + FP) * 100
rec = TP / (TP + FN) * 100
f1 = 2 * (prec/100) * (rec/100) / ((prec/100) + (rec/100)) * 100

print("\nOriginal Transformer:")
print(f"Accuracy: {acc:.2f}")
print(f"Precision: {prec:.2f}")
print(f"Recall: {rec:.2f}")
print(f"F1-measure: {f1:.2f}")

BOS Adjusted Transformer:
Accuracy: 98.23
Precision: 57.88
Recall: 80.69
F1-measure: 67.40

Original Transformer:
Accuracy: 98.90
Precision: 77.92
Recall: 72.03
F1-measure: 74.86


# Semantic Analysis

In [19]:
prediction_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 4679295 entries, 0 to 4679294
Data columns (total 18 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   dataset                    str    
 1   seq_id                     int64  
 2   sequence                   str    
 3   seq_len                    int64  
 4   input_seq                  str    
 5   target_seq                 str    
 6   pred_events                str    
 7   pred_probs                 str    
 8   target_event               int64  
 9   target_event_is_unseen     bool   
 10  target_in_topk_pred        bool   
 11  target_in_pred_pos         float64
 12  target_in_topk_pred_prob   float64
 13  predicted_status           str    
 14  results                    bool   
 15  results_matrix             str    
 16  sequence_predicted_status  str    
 17  sequence_results_matrix    str    
dtypes: bool(3), float64(2), int64(3), str(10)
memory usage: 2.2 GB


In [20]:
prediction_db.head()

,dataset,seq_id,sequence,seq_len,input_seq,target_seq,pred_events,pred_probs,target_event,target_event_is_unseen,target_in_topk_pred,target_in_pred_pos,target_in_topk_pred_prob,predicted_status,results,results_matrix,sequence_predicted_status,sequence_results_matrix
0,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]",[4],"[5, 4, 23, 3, 13, 25, 9, 6, 24]","[0.8432120084762573, 0.15490305423736572, 0.0010511449072510004, 0.0006339344545267522, 9.121267794398591e-05, 3.543875573086552e-05, 3.401046706130728e-05, 1.4385935173777398e-05, 6.067107733542798e-06]",4,False,True,2.0,0.154903,normal,True,NaN,normal,NaN
1,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5]","[5, 23, 4, 3, 25, 13, 6, 9, 24]","[0.9995607733726501, 0.00026684088516049087, 8.573746890760958e-05, 6.649483839282766e-05, 9.479947038926184e-06, 7.3247038017143495e-06, 8.8025836930683e-07, 5.44468775842688e-07, 2.2714397118761553e-07]",5,False,True,1.0,0.999561,normal,True,NaN,normal,NaN
2,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5, 5]","[5, 23, 4, 25, 13, 9, 6, 3, 2]","[0.9908952713012695, 0.006652186159044504, 0.0021032874938100576, 0.00010499305790290236, 0.00010146699787583202, 6.697035860270262e-05, 5.192885873839259e-05, 5.781131221738178e-06, 2.406812654953683e-06]",5,False,True,1.0,0.990895,normal,True,NaN,normal,NaN
3,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5, 5, 5]","[5, 23, 13, 4, 9, 25, 6, 3, 2]","[0.5166937708854675, 0.4775638282299042, 0.0015012141084298491, 0.0013594720512628555, 0.001332766842097044, 0.0012100801104679704, 0.0002610936644487083, 9.44007024372695e-06, 8.358000741282012e-06]",5,False,True,1.0,0.516694,normal,True,NaN,normal,NaN
4,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5, 5, 5, 23]","[23, 9, 5, 13, 25, 6, 4, 24, 8]","[0.9991132616996765, 0.0005334100569598377, 0.00014241792086977512, 0.00013793156540486962, 4.9971320549957454e-05, 1.0128712347068358e-05, 9.66803781921044e-06, 3.934319181553292e-07, 3.332151834456454e-07]",23,False,True,1.0,0.999113,normal,True,NaN,normal,NaN


In [21]:
prediction_db["dataset"].value_counts()

dataset
normal_test      4533578
abnormal_test     105076
train              40641
Name: count, dtype: int64

In [22]:
test_db = prediction_db[
    prediction_db["dataset"].isin(["normal_test", "abnormal_test"])
].copy()

test_db.shape

(4638654, 18)

## Confusion Matrix

In [29]:
row_counts = test_db["results_matrix"].value_counts()

TP = row_counts.get("True Positive", 0)
TN = row_counts.get("True Negative", 0)
FP = row_counts.get("False Positive", 0)
FN = row_counts.get("False Negative", 0)

accuracy = (TP + TN) / (TP + TN + FP + FN) * 100
precision = TP / (TP + FP) * 100 if (TP + FP) > 0 else 0
recall = TP / (TP + FN) * 100 if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Prediction-row-level evaluation")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print(f"Accuracy : {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall   : {recall:.2f}%")
print(f"F1-score : {f1:.2f}%")

Prediction-row-level evaluation
TP: 16182
TN: 4523680
FP: 9898
FN: 88894
Accuracy : 97.87%
Precision: 62.05%
Recall   : 15.40%
F1-score : 24.68%


In [31]:
test_db_ori = prediction_db_ori[
    prediction_db_ori["dataset"].isin(["normal_test", "abnormal_test"])
].copy()

row_counts = test_db_ori["results_matrix"].value_counts()

TP = row_counts.get("True Positive", 0)
TN = row_counts.get("True Negative", 0)
FP = row_counts.get("False Positive", 0)
FN = row_counts.get("False Negative", 0)

accuracy = (TP + TN) / (TP + TN + FP + FN) * 100
precision = TP / (TP + FP) * 100 if (TP + FP) > 0 else 0
recall = TP / (TP + FN) * 100 if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Prediction-row-level evaluation")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print(f"Accuracy : {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall   : {recall:.2f}%")
print(f"F1-score : {f1:.2f}%")

Prediction-row-level evaluation
TP: 16535
TN: 4526550
FP: 7028
FN: 88541
Accuracy : 97.94%
Precision: 70.17%
Recall   : 15.74%
F1-score : 25.71%


In [30]:
sequence_db = test_db.drop_duplicates(
    subset=["dataset", "seq_id"]
).copy()

seq_counts = sequence_db["sequence_results_matrix"].value_counts()

TP = seq_counts.get("True Positive", 0)
TN = seq_counts.get("True Negative", 0)
FP = seq_counts.get("False Positive", 0)
FN = seq_counts.get("False Negative", 0)

accuracy = (TP + TN) / (TP + TN + FP + FN) * 100
precision = TP / (TP + FP) * 100 if (TP + FP) > 0 else 0
recall = TP / (TP + FN) * 100 if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Sequence-level evaluation")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print(f"Accuracy : {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall   : {recall:.2f}%")
print(f"F1-score : {f1:.2f}%")

Sequence-level evaluation
TP: 8601
TN: 546947
FP: 6294
FN: 2046
Accuracy : 98.52%
Precision: 57.74%
Recall   : 80.78%
F1-score : 67.35%


In [32]:
sequence_db_ori = test_db_ori.drop_duplicates(
    subset=["dataset", "seq_id"]
).copy()

seq_counts = sequence_db_ori["sequence_results_matrix"].value_counts()

TP = seq_counts.get("True Positive", 0)
TN = seq_counts.get("True Negative", 0)
FP = seq_counts.get("False Positive", 0)
FN = seq_counts.get("False Negative", 0)

accuracy = (TP + TN) / (TP + TN + FP + FN) * 100
precision = TP / (TP + FP) * 100 if (TP + FP) > 0 else 0
recall = TP / (TP + FN) * 100 if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Sequence-level evaluation")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print(f"Accuracy : {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall   : {recall:.2f}%")
print(f"F1-score : {f1:.2f}%")

Sequence-level evaluation
TP: 7625
TN: 551008
FP: 2233
FN: 3022
Accuracy : 99.07%
Precision: 77.35%
Recall   : 71.62%
F1-score : 74.37%


## Unseen EventID effect on prediction result

In [33]:
unseen_effect = pd.crosstab(
    index=[test_db["dataset"], test_db["target_event_is_unseen"]],
    columns=test_db["results_matrix"],
    normalize="index"
) * 100

unseen_effect.round(2)

results_matrix                        False Negative  False Positive  True Negative  True Positive
dataset       target_event_is_unseen                                                              
abnormal_test False                            88.85            0.00           0.00          11.15
              True                              0.32            0.00           0.00          99.68
normal_test   False                             0.00            0.22          99.78           0.00
              True                              0.00          100.00           0.00           0.00

## Unseen EventID counts

In [35]:
test_db[test_db["target_event_is_unseen"]].groupby("dataset")["target_event"].value_counts().sort_index()

dataset        target_event
abnormal_test  10                59
               11                17
               14               411
               15                31
               17               935
               18                23
               19                55
               20                 1
               21                 8
               22                 8
               24              3448
               26                 1
               27                 2
               28                 4
               29                31
               31                 6
normal_test    17                 2
Name: count, dtype: int64

## Top-k position statistics

In [37]:
topk_hit = test_db[test_db["target_in_topk_pred"] == True].copy()

topk_hit["target_in_pred_pos"].describe()

count    4.612574e+06
mean     1.558555e+00
std      1.372653e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      9.000000e+00
Name: target_in_pred_pos, dtype: float64

In [38]:
topk_hit.groupby("dataset")["target_in_pred_pos"].describe()

,count,mean,std,min,25%,50%,75%,max
dataset,,,,,,,,
abnormal_test,88894.0,2.136781,1.894836,1.0,1.0,1.0,3.0,9.0
normal_test,4523680.0,1.547192,1.357921,1.0,1.0,1.0,1.0,9.0


## True event probability statistics

In [39]:
topk_hit["target_in_topk_pred_prob"].describe()

count    4.612574e+06
mean     7.826831e-01
std      3.681239e-01
min      1.170501e-09
25%      8.587204e-01
50%      9.910880e-01
75%      9.988067e-01
max      9.999996e-01
Name: target_in_topk_pred_prob, dtype: float64

In [40]:
prob_stats = topk_hit.groupby("dataset")["target_in_topk_pred_prob"].agg(
    ["mean", "median", "std", "min", "max"]
)

prob_stats

,mean,median,std,min,max
dataset,,,,,
abnormal_test,0.619973,0.950424,0.447863,6.704621e-09,1.0
normal_test,0.785881,0.991088,0.365659,1.170501e-09,1.0


In [41]:
pos_stats = topk_hit.groupby("dataset")["target_in_pred_pos"].agg(
    ["mean", "median", "std", "min", "max"]
)

pos_stats

,mean,median,std,min,max
dataset,,,,,
abnormal_test,2.136781,1.0,1.894836,1.0,9.0
normal_test,1.547192,1.0,1.357921,1.0,9.0


## EventID frequency by result type

In [43]:
test_db.groupby("results_matrix")["target_event"].value_counts()

results_matrix  target_event
False Negative  5                 24457
                23                17309
                13                14855
                6                 10378
                4                  9869
                25                 5103
                3                  4786
                9                  2065
                8                    53
                24                   15
                2                     3
                22                    1
False Positive  6                  3685
                1                  1845
                7                  1739
                8                  1682
                23                  507
                2                   304
                5                   102
                9                    20
                13                   12
                17                    2
True Negative   5               1211259
                23              1157227
           

## Dataset summary

In [ ]:
test_db = prediction_db[
    prediction_db["dataset"].isin(["normal_test", "abnormal_test"])
].copy()

In [48]:
summary_table = test_db.groupby("dataset").agg(
    unseen_event=("target_event_is_unseen", "sum"),
    topk_hit_rate=("target_in_topk_pred", "mean"),
    prediction_position=("target_in_pred_pos", "mean"),
    mean_target_prob=("target_in_topk_pred_prob", "mean"),
    median_target_prob=("target_in_topk_pred_prob", "median"),
)

summary_table["topk_hit_rate"] = summary_table["topk_hit_rate"] * 100
summary_table.round(2)

,unseen_event,topk_hit_rate,prediction_position,mean_target_prob,median_target_prob
dataset,,,,,
abnormal_test,5040,84.60,2.14,0.62,0.95
normal_test,2,99.78,1.55,0.79,0.99


In [47]:
test_db.head(5)

,dataset,seq_id,sequence,seq_len,input_seq,target_seq,pred_events,pred_probs,target_event,target_event_is_unseen,target_in_topk_pred,target_in_pred_pos,target_in_topk_pred_prob,predicted_status,results,results_matrix,sequence_predicted_status,sequence_results_matrix
40641,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]",[4],"[4, 3, 9, 5, 25, 24, 22, 12, 2]","[0.9903373718261719, 0.009661218151450157, 6.329992174869403e-07, 2.7030228011426516e-07, 2.1788400772493333e-07, 1.7994237566654192e-07, 2.916568142552478e-08, 2.8556975451010658e-08, 2.150493294550415e-08]",4,False,True,1.0,0.990337,normal,True,True Negative,normal,True Negative
40642,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3]","[3, 4, 5, 25, 23, 26, 24, 2, 28]","[0.9997363686561584, 0.00018246128456667066, 7.214649667730555e-05, 6.90634533384582e-06, 3.9705611243334715e-07, 3.125783223367762e-07, 1.6053230922352668e-07, 1.4491354249912547e-07, 1.0573919695389122e-07]",3,False,True,1.0,0.999736,normal,True,True Negative,normal,True Negative
40643,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3, 4]","[4, 9, 3, 24, 10, 5, 25, 6, 2]","[0.9999996423721313, 2.5044428753062675e-07, 1.1106849484576742e-07, 1.3098173212711117e-08, 1.059248511126043e-08, 1.0226953506276004e-08, 6.82894674142176e-09, 5.357434051944665e-09, 4.159580679896635e-09]",4,False,True,1.0,1.000000,normal,True,True Negative,normal,True Negative
40644,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3, 4, 9]","[23, 25, 9, 5, 3, 6, 4, 13, 15]","[0.5589843988418579, 0.4388941526412964, 0.0009988080710172653, 0.0007600008975714445, 0.00022740353597328067, 3.102449409198016e-05, 2.6064000849146396e-05, 1.656256426940672e-05, 8.248433914559428e-06]",9,False,True,3.0,0.000999,normal,True,True Negative,normal,True Negative
40645,normal_test,0,2 1 1 1 5 3 4 5 5 3 4 3 4 9 6 25 6 6 6 25 6 25 6 6 25 6 23 23 23 13 24 13 13,33,"[2, 1, 1, 1, 5, 3, 4, 5, 5, 3]","[4, 3, 4, 9, 6]","[23, 9, 25, 4, 13, 6, 24, 12, 8]","[0.8473748564720154, 0.1509784460067749, 0.001563208643347025, 6.786578160244972e-05, 4.2939518607454374e-06, 3.296773684269283e-06, 1.3069163742329692e-06, 9.290149023399863e-07, 8.16922067770065e-07]",6,False,True,6.0,0.000003,normal,True,True Negative,normal,True Negative
